<a href="https://colab.research.google.com/github/rithakith/sri-lankan-news-ensemble/blob/main/News_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Load the source CSV

### 1.1 Import libraries and pin the source snapshot

In [ ]:
from pathlib import Path
import pandas as pd
import csv, io, json, zipfile, urllib.request

# Exact upstream snapshot used to prepare our dataset.
COMMIT = '8a14da7c922131fcf6adc4f70726cabcb1c518d8'
DATA_DIR = Path('/content/drive/MyDrive/News_Preprocessing')

DATA_DIR.mkdir(parents=True, exist_ok=True)

archive_path = DATA_DIR / 'lk_news_snapshot.zip'
INPUT_PATH = DATA_DIR / 'news_english.csv'

### 1.2 Retrieve and construct the source CSV


In [ ]:

# Download the archive only if needed, then construct the English CSV.

if not INPUT_PATH.exists():

    # Check whether we already have a valid ZIP archive in Google Drive
    if not zipfile.is_zipfile(archive_path):
        print('Downloading the pinned source archive...', flush=True)

        url = f'https://codeload.github.com/nuuuwan/lk_news/zip/{COMMIT}'
        urllib.request.urlretrieve(url, archive_path)

        # Verify the downloaded archive
        if not zipfile.is_zipfile(archive_path):
            raise ValueError("Downloaded file is not a valid ZIP archive")

    print('Constructing the English CSV from metadata and article text...', flush=True)

    with zipfile.ZipFile(archive_path) as z:
        names = z.namelist()

        manifest_path = next(
            n for n in names
            if n.endswith('/data/lk_news/docs_all.tsv')
        )

        metadata = list(
            csv.DictReader(
                io.StringIO(z.read(manifest_path).decode('utf-8-sig')),
                delimiter='\t'
            )
        )

        paths = {
            n.split('/')[-2]: n.rsplit('/', 1)[0]
            for n in names if n.endswith('/doc.json')
        }

        records = []

        for m in metadata:
            if m['lang'] == 'en':
                p = paths[m['doc_id']]

                r = json.loads(z.read(p + '/doc.json'))
                r['text'] = z.read(p + '/doc.txt').decode('utf-8')

                records.append(r)

    # Save the constructed CSV to Google Drive
    pd.DataFrame(records).to_csv(
        INPUT_PATH,
        index=False,
        encoding='utf-8'
    )

    print(f'CSV saved to: {INPUT_PATH}')

    del records, metadata, paths, names

else:
    print('English CSV already exists in Google Drive. Skipping construction.')


Constructing the English CSV from metadata and article text...
CSV saved to: /content/drive/MyDrive/News_Preprocessing/news_english.csv


### 1.3 Load and inspect the articles

In [ ]:
# Read the source CSV, check the expected 51,590 records and preview metadata without showing large article bodies.

df = pd.read_csv(INPUT_PATH, keep_default_na=False)
assert len(df) == 51590
print(f'Loaded {len(df):,} articles and {len(df.columns)} columns.')
display(df.drop(columns=['text']).head())


Loaded 51,590 articles and 10 columns.


,doc_type,doc_id,num,date_str,description,url_metadata,lang,newspaper_id,time_ut
0,lk_news,2026-11-20-dailymirrorlk-b864884a,dailymirrorlk-b864884a,2026-11-20,Parliament approves second reading of Budget 2027,https://www.dailymirror.lk/breaking-news/Parli...,en,dailymirrorlk,1.795182e+09
1,lk_news,2026-11-20-dailymirrorlk-b0292801,dailymirrorlk-b0292801,2026-11-20,CBSL introduces new rules on vehicle valuation...,https://www.dailymirror.lk/breaking-news/CBSL-...,en,dailymirrorlk,1.795182e+09
2,lk_news,2026-11-20-dailymirrorlk-fe9b1f03,dailymirrorlk-fe9b1f03,2026-11-20,Afternoon thundershowers expected across sever...,https://www.dailymirror.lk/breaking-news/After...,en,dailymirrorlk,1.795179e+09
3,lk_news,2026-11-20-dailymirrorlk-0ab016f4,dailymirrorlk-0ab016f4,2026-11-20,Poland says Russian missile hit its territory ...,https://www.dailymirror.lk/international/Polan...,en,dailymirrorlk,1.795174e+09
4,lk_news,2026-11-20-dailymirrorlk-123c85c9,dailymirrorlk-123c85c9,2026-11-20,FBI team visits Sri Lanka for scam probe,https://www.dailymirror.lk/breaking-news/FBI-t...,en,dailymirrorlk,1.795172e+09


### 2. Cleaning the CSV

### 2.1 Configure cleaning rules

Set the minimum word count and Latin-script ratio

In [ ]:
import hashlib
import re
import unicodedata

MIN_WORDS = 50
MIN_LATIN_RATIO = 0.80
original_columns = df.columns.tolist()
required = {'doc_id', 'date_str', 'lang', 'text'}
assert required.issubset(df.columns), f"Missing columns: {required - set(df.columns)}"
assert df['doc_id'].is_unique, "Review duplicate document IDs before continuing."


### 2.2 Define Helper Functions for Text Quality

Define two reusable functions to improve dataset quality:

- **`normalized_text_hash()`:** Normalizes Unicode, whitespace, and capitalization before generating a SHA-256 hash to identify duplicate articles.
- **`latin_letter_ratio()`:** Calculates the proportion of Latin letters in an article to help filter text predominantly written in other scripts.

These functions support duplicate detection and language-based filtering during preprocessing.

In [ ]:
# 2.2 Define reusable text-quality helpers: Create normalized hashes to identify exact duplicates after whitespace and Unicode normalization, and estimate Latin-letter proportions.
def normalized_text_hash(text):
    normalized = unicodedata.normalize('NFKC', str(text))
    normalized = re.sub(r'\s+', ' ', normalized).strip().casefold()
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()

def latin_letter_ratio(text):
    letters = [char for char in str(text) if char.isalpha()]
    if not letters:
        return 0.0
    latin_letters = sum('LATIN' in unicodedata.name(char, '') for char in letters)
    return latin_letters / len(letters)

### 2.3 Calculate article-quality flags

Sort deterministically and flag short text, language concerns, replacement characters and normalized exact duplicates. An article may have more than one flag.

In [ ]:
working = df.sort_values(['date_str', 'doc_id'], kind='stable').copy()
word_counts = working['text'].str.split().str.len()
latin_ratios = working['text'].map(latin_letter_ratio)
text_hashes = working['text'].map(normalized_text_hash)

flags = pd.DataFrame({
    'not_tagged_english': working['lang'].ne('en'),
    'short_text': word_counts.lt(MIN_WORDS),
    'language_review': latin_ratios.lt(MIN_LATIN_RATIO),
    'replacement_character': working['text'].str.contains('\ufffd', regex=False),
    'normalized_exact_duplicate': text_hashes.duplicated(keep='first'),
}, index=working.index)


### 2.4 Summarize and filter flagged articles

Count overlapping quality flags, remove rows flagged by any rule and report the number retained.

In [ ]:
summary = flags.sum().rename('flagged_articles').to_frame()
print("Flag counts can overlap:")
display(summary)

remove_mask = flags.any(axis=1)
cleaned_df = working.loc[~remove_mask, original_columns].reset_index(drop=True)
print(f"Input:   {len(df):,}")
print(f"Removed: {int(remove_mask.sum()):,} unique articles")
print(f"Kept:    {len(cleaned_df):,}")


Flag counts can overlap:


,flagged_articles
not_tagged_english,0
short_text,942
language_review,84
replacement_character,2
normalized_exact_duplicate,34


Input:   51,590
Removed: 1,062 unique articles
Kept:    50,528


### 2.5 Remove the flagged articles and save the cleaned CSV


In [ ]:
assert cleaned_df.columns.tolist() == original_columns
assert cleaned_df['doc_id'].is_unique
assert cleaned_df['lang'].eq('en').all()
assert not cleaned_df['text'].map(normalized_text_hash).duplicated().any()
assert len(cleaned_df) + int(remove_mask.sum()) == len(df)

OUTPUT_PATH = DATA_DIR / 'news_english_cleaned.csv'
cleaned_df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')
saved = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
pd.testing.assert_frame_equal(saved, cleaned_df)
print(f"Saved and verified: {OUTPUT_PATH.resolve()}")


Saved and verified: /content/drive/MyDrive/News_Preprocessing/news_english_cleaned.csv


### 4. Full-dataset zero-shot pseudo-labelling

### 4.1 Load the cleaned csv

In [ ]:
from pathlib import Path
import pandas as pd
import torch
import numpy as np
import json, hashlib, os, time
from transformers import pipeline
from tqdm.auto import tqdm
INPUT_PATH = DATA_DIR / 'news_english_cleaned.csv'
df = pd.read_csv(INPUT_PATH, keep_default_na=False)
assert len(df) == 50528 and df['doc_id'].is_unique
assert df['text'].str.strip().ne('').all()
print(f'Loaded {len(df):,} cleaned articles; preserving {len(df.columns)} original columns.')


Loaded 50,528 cleaned articles; preserving 10 original columns.


### 4.2 Define News Categories and Alternative Topic Phrases


- TOPICS: Maps each final category to a descriptive phrase used by the classification model.
- ALIASES: Maps alternative phrases (e.g., "cricket", "football") to their corresponding categories (e.g., sport).

Multiple phrases help capture different subjects within a category. The highest matching phrase score is used as the category score.

In [ ]:

# Zero-shot classification model
MODEL_ID = 'facebook/bart-large-mnli'

# Final news categories and their descriptions
TOPICS = {
    'politics_governance': 'politics and government',
    'business_economy': 'business and economics',
    'crime_justice': 'crime and court cases',
    'sport': 'sports',
    'health_healthcare': 'Health & Healthcare',
    'weather_environment_disasters': 'Weather, Environment & Disasters',
    'tourism_travel': 'Tourism & Travel',
    'transport': 'transport and vehicles',
    'telecommunications': 'telecommunications and internet connectivity',
    'information_technology': 'software, computing or artificial intelligence',
    'education': 'education',
    'entertainment': 'entertainment and show business',
    'other': 'lifestyle, culture, religion or general-interest news',
}

# Alternative topic phrases mapped to the final categories
ALIASES = {
    'politics': 'politics_governance',
    'government and diplomacy': 'politics_governance',
    'business': 'business_economy',
    'economics': 'business_economy',
    'crime': 'crime_justice',
    'legal proceedings': 'crime_justice',
    'sports': 'sport',
    'cricket': 'sport',
    'football': 'sport',
    'medicine and healthcare': 'health_healthcare',
    'weather': 'weather_environment_disasters',
    'environmental protection and pollution': 'weather_environment_disasters',
    'natural disasters': 'weather_environment_disasters',
    'transport': 'transport',
    'vehicles': 'transport',
    'telecommunications': 'telecommunications',
    'information technology': 'information_technology',
    'education': 'education',
    'entertainment': 'entertainment',
    'music': 'entertainment',
    'films and television': 'entertainment',
    'tourism and travel': 'tourism_travel',
    'hotels and holiday destinations': 'tourism_travel',
    'religion and spirituality': 'other',
    'lifestyle and human interest': 'other',
}

# Hypothesis template used by the zero-shot classifier
TEMPLATE = 'This text is about {}.'


In [ ]:
# Define the score columns for all 13 news categories
score_columns = ['score_' + topic for topic in TOPICS]

print(score_columns)

['score_politics_governance', 'score_business_economy', 'score_crime_justice', 'score_sport', 'score_health_healthcare', 'score_weather_environment_disasters', 'score_tourism_travel', 'score_transport', 'score_telecommunications', 'score_information_technology', 'score_education', 'score_entertainment', 'score_other']


### 4.3 Load the zero-shot model on GPU


In [ ]:
# 4.4 Load the zero-shot model on GPU: Use the pinned model revision, half-precision weights and an initial batch size of 16.
MODEL_REVISION = 'd7645e127eaf1aefc7862fd59a17a5aa8558b8ce'
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before the full run.'
classifier = pipeline('zero-shot-classification', model=MODEL_ID,
                      revision=MODEL_REVISION, device=0, torch_dtype=torch.float16)
REVISION = getattr(classifier.model.config, '_commit_hash', None) or 'unknown'
BATCH_SIZE = 16
print('Device:', classifier.device, '| model revision:', REVISION)


config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device: cuda:0 | model revision: unknown


### 4.4 Define topic prediction and review flags

### 4.4.1 Configure Review Thresholds

- **Minimum score (0.70):** The highest category score must be at least 0.70.
- **Minimum margin (0.20):** The difference between the two highest category scores must be at least 0.20.

Predictions that fail either condition are flagged for review.

In [ ]:
MIN_SCORE = 0.70
MIN_MARGIN = 0.20

### 4.4.2 Prepare Article Inputs

Combine each article's description (used as the headline) with the first 200 words of its body to create the input for zero-shot classification.

In [ ]:
def prepare_inputs(frame):
    inputs = [
        str(title) + '\n' + ' '.join(str(text).split()[:200])
        for title, text in zip(frame['description'], frame['text'])
    ]

    return inputs

### 4.5.3 Run Zero-Shot Classification

Use BART-MNLI to independently score each candidate topic phrase. If GPU memory is insufficient, automatically reduce the batch size and retry.

In [ ]:
def run_classifier(inputs):
    global BATCH_SIZE

    while True:
        try:
            results = classifier(
                inputs,
                candidate_labels=list(ALIASES),
                hypothesis_template=TEMPLATE,
                multi_label=True,
                batch_size=BATCH_SIZE
            )
            return results

        except torch.cuda.OutOfMemoryError:
            if BATCH_SIZE == 1:
                raise

            BATCH_SIZE = max(1, BATCH_SIZE // 2)
            torch.cuda.empty_cache()

            print(
                'Reduced GPU batch size to',
                BATCH_SIZE,
                flush=True
            )

### 4.5.4 Aggregate Alias Scores and Rank Topics

Map candidate phrases to the 13 final news categories. Use the highest alias score as each category's score, then rank categories from highest to lowest.

In [ ]:
def rank_topics(result):
    alias_scores = dict(
        zip(result['labels'], result['scores'])
    )

    scores = {
        topic: max(
            alias_scores[a]
            for a, group in ALIASES.items()
            if group == topic
        )
        for topic in TOPICS
    }

    ranked = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return alias_scores, scores, ranked

### 4.5.5 Generate Prediction Records and Review Flags

Store the predicted category, highest-matching phrase, confidence-related scores, review flag, and model metadata for each article. Retain scores for all 13 categories.

In [ ]:
def create_prediction_record(doc_id, result):

    alias_scores, scores, ranked = rank_topics(result)

    predicted_topic = ranked[0]

    score = scores[ranked[0]]
    second_score = scores[ranked[1]]

    margin = score - second_score

    needs_review = (
        score < MIN_SCORE
        or margin < MIN_MARGIN
    )

    matched_phrase = max(
        (
            a for a, group in ALIASES.items()
            if group == predicted_topic
        ),
        key=alias_scores.get
    )

    row = {
        'doc_id': doc_id,
        'predicted_topic': predicted_topic,
        'matched_phrase': matched_phrase,
        'teacher_score': score,
        'teacher_margin': margin,
        'needs_review': needs_review,
        'label_source': 'zero_shot_pseudo_label',
        'teacher_model': MODEL_ID,
        'teacher_revision': REVISION,
        'taxonomy_version': 'v8',
        'input_policy': 'headline_plus_first_200_body_words',
        'scoring_policy': 'independent_alias_scores_max_per_topic',
    }

    row.update({
        'score_' + topic: scores[topic]
        for topic in TOPICS
    })

    return row

### 4.5.6 Define the Main Prediction Function

Combine input preparation, zero-shot classification, and prediction-record generation into one reusable function. Validate the category configuration before classification.

In [ ]:
def predict_topics(frame):

    inputs = prepare_inputs(frame)

    results = run_classifier(inputs)

    # The pipeline may return a dictionary for one input.
    if isinstance(results, dict):
        results = [results]

    rows = []

    for doc_id, result in zip(frame['doc_id'], results):

        row = create_prediction_record(doc_id, result)

        rows.append(row)

    return pd.DataFrame(rows)


# Validate category configuration
assert len(TOPICS) == 13
assert set(ALIASES.values()) == set(TOPICS)

### 4.5 Choose checkpoint and output directories


In [ ]:
# Saves each completed 100-row chunk atomically; safe to rerun with the same input/config.
if Path('/content/drive/MyDrive').exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    PARTS = DATA_DIR / 'checkpoints'
    PARTS.mkdir(exist_ok=True)
    OUTPUT = DATA_DIR / 'labelled_zero_shot_bart.csv'


### 4.6 Record reproducible run settings

Hash the input CSV and record the model, taxonomy and scoring settings.

In [ ]:
input_hash = hashlib.sha256()
with INPUT_PATH.open('rb') as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b''):
        input_hash.update(block)
settings = {'model': MODEL_ID, 'revision': REVISION, 'topics': TOPICS, 'aliases': ALIASES,
            'template': TEMPLATE, 'taxonomy': 'v8', 'input_sha256': input_hash.hexdigest(),
            'input_policy': 'headline_plus_first_200_body_words', 'precision': 'float16',
            'min_score': MIN_SCORE, 'min_margin': MIN_MARGIN,
            'scoring_policy': 'independent_alias_scores_max_per_topic', 'chunk_size': 100}
config_id = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()
manifest_path = DATA_DIR / 'run_settings.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == settings, 'Input/config changed: use a new run directory.'
else:
    manifest_path.write_text(json.dumps(settings, indent=2))


### 4.7 Label articles in resumable chunks

Run 100 articles per chunk, skipping verified prior checkpoints

In [ ]:
# Limit classification to the first 10,000 articles
LIMIT = 10000
subset_df = df.head(LIMIT).copy()

started = time.monotonic()

# Process articles in chunks of 100
for start in tqdm(
    range(0, len(subset_df), 100),
    desc='Classifying first 10,000 articles'
):

    frame = subset_df.iloc[start:start + 100]
    part = PARTS / f'part_{start:06d}.csv'

    # Skip previously completed and validated chunks
    if part.exists():
        saved = pd.read_csv(part, keep_default_na=False)

        assert saved['doc_id'].tolist() == frame['doc_id'].tolist()
        assert saved['run_config_id'].eq(config_id).all()
        assert np.isfinite(saved[score_columns].to_numpy()).all()

        continue

    # Classify the current chunk
    result = predict_topics(frame)

    # Validate predictions
    assert len(result) == len(frame)
    assert result['doc_id'].tolist() == frame['doc_id'].tolist()
    assert np.isfinite(result[score_columns].to_numpy()).all()

    result['run_config_id'] = config_id

    # Save checkpoint
    temp = part.with_suffix('.csv.tmp')
    result.to_csv(temp, index=False)
    os.replace(temp, part)

    completed = min(start + 100, len(subset_df))

    if completed % 1000 == 0 or completed == len(subset_df):
        print(
            f'Checkpointed {completed:,}/{len(subset_df):,} articles; '
            f'elapsed {(time.monotonic() - started)/60:.1f} min',
            flush=True
        )


# ---------------------------------------------------
# Combine all completed checkpoints into one CSV file
# ---------------------------------------------------

all_parts = []

for start in range(0, len(subset_df), 100):

    part = PARTS / f'part_{start:06d}.csv'

    saved = pd.read_csv(part, keep_default_na=False)

    # Verify each checkpoint before combining
    frame = subset_df.iloc[start:start + 100]
    assert saved['doc_id'].tolist() == frame['doc_id'].tolist()
    assert saved['run_config_id'].eq(config_id).all()
    assert np.isfinite(saved[score_columns].to_numpy()).all()

    all_parts.append(saved)

# Combine all predictions
final_df = pd.concat(all_parts, ignore_index=True)

# Verify final dataset
assert len(final_df) == len(subset_df)
assert final_df['doc_id'].tolist() == subset_df['doc_id'].tolist()

# Save final CSV to Google Drive
OUTPUT_10K = DATA_DIR / 'news_english_cleaned_top_10000_labelled.csv'

temp_output = OUTPUT_10K.with_suffix('.csv.tmp')
final_df.to_csv(temp_output, index=False)
os.replace(temp_output, OUTPUT_10K)

print(f'\nSaved {len(final_df):,} classified articles')
print(f'Output: {OUTPUT_10K}')
print(f'Total elapsed time: {(time.monotonic() - started)/60:.1f} minutes')

Classifying first 10,000 articles:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Checkpointed 1,000/10,000 articles; elapsed 5.5 min
Checkpointed 2,000/10,000 articles; elapsed 10.7 min
Checkpointed 3,000/10,000 articles; elapsed 16.0 min
Checkpointed 4,000/10,000 articles; elapsed 21.4 min
Checkpointed 5,000/10,000 articles; elapsed 26.7 min
Checkpointed 6,000/10,000 articles; elapsed 32.0 min
Checkpointed 7,000/10,000 articles; elapsed 37.3 min
Checkpointed 8,000/10,000 articles; elapsed 42.7 min
Checkpointed 9,000/10,000 articles; elapsed 47.9 min
Checkpointed 10,000/10,000 articles; elapsed 53.3 min

Saved 10,000 classified articles
Output: /content/drive/MyDrive/News_Preprocessing/news_english_cleaned_top_10000_labelled.csv
Total elapsed time: 53.4 minutes


### 4.8 print the labelled dataset

In [ ]:
print(final_df.columns.tolist())

print("Contains article text:", 'text' in final_df.columns)
print("Contains description:", 'description' in final_df.columns)

['doc_id', 'predicted_topic', 'matched_phrase', 'teacher_score', 'teacher_margin', 'needs_review', 'label_source', 'teacher_model', 'teacher_revision', 'taxonomy_version', 'input_policy', 'scoring_policy', 'score_politics_governance', 'score_business_economy', 'score_crime_justice', 'score_sport', 'score_health_healthcare', 'score_weather_environment_disasters', 'score_tourism_travel', 'score_transport', 'score_telecommunications', 'score_information_technology', 'score_education', 'score_entertainment', 'score_other', 'run_config_id']
Contains article text: False
Contains description: False


4.9 Load the saved predictions

In [ ]:
# Load the 10,000 classified articles
PREDICTIONS_PATH = DATA_DIR / 'news_english_cleaned_top_10000_labelled.csv'

predictions_df = pd.read_csv(
    PREDICTIONS_PATH,
    keep_default_na=False
)

print("Predictions loaded:", len(predictions_df))

Predictions loaded: 10000


### Merge predictions with original article data

In [ ]:
# Select the same first 10,000 original articles
subset_df = df.head(10000).copy()

# Verify article IDs match
assert predictions_df['doc_id'].tolist() == subset_df['doc_id'].tolist()

# Merge original article data with predicted labels
labelled_df = subset_df.merge(
    predictions_df,
    on='doc_id',
    how='left',
    validate='one_to_one'
)

# Verify the merge
assert len(labelled_df) == 10000
assert labelled_df['text'].notna().all()
assert labelled_df['predicted_topic'].notna().all()

print("Total articles:", len(labelled_df))
print("Contains text:", 'text' in labelled_df.columns)
print("Contains description:", 'description' in labelled_df.columns)

display(labelled_df[['doc_id', 'description', 'predicted_topic', 'teacher_score', 'needs_review']].head())

Total articles: 10000
Contains text: True
Contains description: True


,doc_id,description,predicted_topic,teacher_score,needs_review
0,2022-07-12-newsfirstlk-c83f4c6f,LPL 2022 schedule announced; 24 matches in total,sport,0.973931,False
1,2023-10-07-newsfirstlk-ddd24a07,ICC CWC 2023 - Match Preview: Sri Lanka v Sout...,sport,0.982541,False
2,2023-10-18-dbsjeyarajcom-3a451bcb,President Wickremesinghe Likely to Make Drasti...,politics_governance,0.912358,True
3,2023-10-18-dbsjeyarajcom-d4bba182,No Moves on to Abolish the Executive Presiden...,politics_governance,0.959269,True
4,2023-10-19-dbsjeyarajcom-15c6a51b,If No One Comes Forward to Contest the Next P...,politics_governance,0.975460,True


In [ ]:
labelled_df = labelled_df.rename(columns={
    'teacher_score': 'classification_score',
    'teacher_margin': 'score_margin'
})

In [ ]:
print(labelled_df.head())

  doc_type                             doc_id                     num  \
0  lk_news    2022-07-12-newsfirstlk-c83f4c6f    newsfirstlk-c83f4c6f   
1  lk_news    2023-10-07-newsfirstlk-ddd24a07    newsfirstlk-ddd24a07   
2  lk_news  2023-10-18-dbsjeyarajcom-3a451bcb  dbsjeyarajcom-3a451bcb   
3  lk_news  2023-10-18-dbsjeyarajcom-d4bba182  dbsjeyarajcom-d4bba182   
4  lk_news  2023-10-19-dbsjeyarajcom-15c6a51b  dbsjeyarajcom-15c6a51b   

     date_str                                        description  \
0  2022-07-12   LPL 2022 schedule announced; 24 matches in total   
1  2023-10-07  ICC CWC 2023 - Match Preview: Sri Lanka v Sout...   
2  2023-10-18  President Wickremesinghe Likely to Make Drasti...   
3  2023-10-18  No Moves on to Abolish the Executive  Presiden...   
4  2023-10-19  If No One Comes Forward to  Contest the Next P...   

                                        url_metadata lang   newspaper_id  \
0  https://english.newsfirst.lk/2022/07/12/lpl-20...   en    newsfirstlk   


In [ ]:
FINAL_OUTPUT = DATA_DIR / 'news_english_top_10000_final.csv'

labelled_df.to_csv(FINAL_OUTPUT, index=False)